## LSTM을 이용한 네이버 영화 리뷰 분류

### 0. 자연어 처리 (Natural Language Processing, NLP)

* 자연어: 일상 생활에서 사용하는 언어
* 자연어 처리: 자연어의 의미를 분석하여 컴퓨터가 처리할 수 있도록 하는 일
    - Ex. 음석 인식, 내용 요약, 번역, 감성 분석, 텍스트 분류(스팸 메일 분류 등)

### 1. 네이버 영화 리뷰 데이터 이해 및 전처리

In [ ]:
#!pip install konlpy

In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import urllib.request
from konlpy.tag import Okt
from tqdm import tqdm
from collections import Counter


In [ ]:
## 토큰화 및 vocabulary 생성 완료 데이터 로드
with open("Data/ratings/ratings.pkl", "rb") as f:
    data = pickle.load(f)

test_data = data['test_data']
train_data = data['train_data']
X_test = data['X_test']
X_train = data['X_train']
vocab = data['vocab']

#### 1) 데이터 로드

In [ ]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", filename="./Data/ratings/ratings_train.txt")  # train 데이터 다운로드
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt", filename="./Data/ratings/ratings_test.txt")    # test 데이터 다운로드

In [ ]:
# pandas를 이용해 데이터 로드
train_data = pd.read_table('./Data/ratings/ratings_train.txt')
test_data = pd.read_table('./Data/ratings/ratings_test.txt')

In [ ]:
print('훈련용 리뷰 개수 :',len(train_data)) # 훈련용 리뷰 개수 출력

In [ ]:
train_data.head()


In [ ]:
test_data.head()

#### 2) 데이터 정제하기

In [ ]:
# 결측치 확인
train_data.isnull().sum()

In [ ]:
train_data = train_data.dropna(axis=0) # Null 값이 존재하는 행 제거

In [ ]:
# document 열과 label 열의 중복을 제외한 값의 개수
train_data['document'].nunique(), train_data['label'].nunique()

In [ ]:
# document 열의 중복 제거
train_data = train_data.drop_duplicates(subset=['document'])

In [ ]:
train_data.shape

In [ ]:
print((train_data.label == 0).sum())
print((train_data.label == 1).sum())

In [ ]:
train_data = train_data.dropna(axis=0)

In [ ]:
## test 데이터 전처리
test_data.drop_duplicates(subset = ['document']) # document 열에서 중복인 내용이 있다면 중복 제거
test_data['document'] = test_data['document'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]","", regex=True) # 한글만 남기기
test_data['document'] = test_data['document'].str.replace('^ +', "", regex=True) # 공백은 empty 값으로 변경
test_data = test_data.replace({'document': ''}, np.nan) # 공백은 Null 값으로 변경
test_data = test_data.dropna(axis=0) # Null 값 제거
print('전처리 후 테스트용 샘플의 개수 :',len(test_data))

**정규 표현식(Regular Expression)**

* 정규 표현식: 특정 규칙을 가진 문자열을 찾거나, 바꾸거나, 지우기 위한 표현 방법
    - Ex) "숫자만 찾기", "한글이 아닌 문자는 모두 지우기"
* 파이썬에서는 `re` 모듈을 사용하고, pandas에서는 `.str.replace(패턴, 바꿀 문자열, regex=True)`로 열 전체에 적용할 수 있음
* 자주 쓰는 기호

| 기호 | 의미 | 예시 |
|---|---|---|
| `[ ]` | 대괄호 안의 문자 중 **하나** (문자 클래스) | `[abc]`: a 또는 b 또는 c |
| `-` | 문자 클래스 안에서 범위 | `[a-z]`: 소문자 하나, `[0-9]`: 숫자 하나 |
| `[^ ]` | 문자 클래스 **안**의 `^`: 대괄호 안의 문자가 **아닌** 것 | `[^0-9]`: 숫자가 아닌 문자 하나 |
| `+` | 앞의 문자가 **1개 이상** 반복 | `[0-9]+`: 1자리 이상의 숫자 |
| `*` | 앞의 문자가 **0개 이상** 반복 | `a*`: a가 없거나 여러 개 |
| `^` | 문자 클래스 **밖**의 `^`: 문자열의 **시작** | `^ +`: 문자열 앞쪽의 공백들 |
| `.` | 아무 문자 하나 | `a.c`: abc, a1c, ... |

* 한글 범위: `ㄱ-ㅎ` (자음), `ㅏ-ㅣ` (모음), `가-힣` (완성형 한글) $\rightarrow$ `[ㄱ-ㅎㅏ-ㅣ가-힣]`은 한글 한 글자

In [ ]:
text = '영화 완전 재밌었어요!! 10점 만점에 10점 ㅋㅋ ^^'

print(re.findall('[0-9]+', text))           # 숫자 찾기
print(re.findall('[ㄱ-ㅎㅏ-ㅣ가-힣]+', text)) # 한글 찾기

In [ ]:
print(re.sub('[^ㄱ-ㅎㅏ-ㅣ가-힣 ]', '', text)) # 한글과 공백이 아닌 문자를 모두 제거 (= 한글과 공백만 남김)
print(re.sub('^ +', '', '    앞쪽 공백 제거'))   # 문자열 시작 부분의 공백 제거

* `re.findall(패턴, 문자열)`: 패턴에 맞는 부분을 모두 찾아 리스트로 반환
* `re.sub(패턴, 바꿀 문자열, 문자열)`: 패턴에 맞는 부분을 바꿀 문자열로 교체
* 아래에서는 같은 패턴을 pandas의 `.str.replace()`에 넣어 `document` 열 전체에 적용함

In [ ]:
# 한글과 공백을 제외하고 모두 제거
# []: 문자 클래스
# ^: not
train_data['document'] = train_data['document'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]","", regex=True)
train_data.head()

In [ ]:
# ^: 문자열의 시작
# +: 1개 이상 연속된 문자
train_data['document'] = train_data['document'].str.replace('^ +', "", regex=True) # white space 데이터를 empty value로 변경
train_data = train_data.replace({'document': ''}, np.nan)
print(train_data.isnull().sum())

#### 3) 토큰화(Tokenization)

* 토큰화: 자연어 데이터를 토큰(token)이라 불리는 단위로 나누는 작업
* 예시
    - 입력: Time is an illusion. Lunchtime double so!
    - 출력 : "Time", "is", "an", "illustion", "Lunchtime", "double", "so", "!"
* 영어에서는 단어 단위, 한국어에서는 형태소 단위를 많이 사용함

**불용어 제거**
* 불용어(Stopwords): 조사, 접미사 같이 문장에서는 자주 등장하지만 실제 의미 분석을 하는데는 거의 기여하는 바가 토큰들

In [ ]:
stopwords = ['도', '는', '다', '의', '가', '이', '은', '한', '에', '하', '고', '을', '를', '인', '듯', '과', '와', '네', '들', '듯', '지', '임', '게']

In [ ]:
okt = Okt()
okt.morphs('와 이런 것도 영화라고 차라리 뮤직비디오를 만드는 게 나을 뻔')

In [ ]:
X_train = []
for sentence in tqdm(train_data['document']):
    tokenized_sentence = okt.morphs(sentence) # 토큰화
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords] # 불용어 제거
    X_train.append(stopwords_removed_sentence)

In [ ]:
train_data.head(5)

In [ ]:
print(X_train[:5])


In [ ]:
X_test = []
for sentence in tqdm(test_data['document']):
    tokenized_sentence = okt.morphs(sentence) # 토큰화
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords] # 불용어 제거
    X_test.append(stopwords_removed_sentence)

#### 4) 단어 집합(Vocabulary) 만들기

In [ ]:
word_list = []
for sent in X_train:
    for word in sent:
      word_list.append(word)

word_counts = Counter(word_list)
print('총 단어수 :', len(word_counts))

In [ ]:
print('훈련 데이터에서의 단어 \'영화\'의 등장 횟수 :', word_counts['영화'])
print('훈련 데이터에서의 단어 \'공감\'의 등장 횟수 :', word_counts['공감'])


In [ ]:
word_counts

In [ ]:
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
print('등장 빈도수 상위 10개 단어')
print(vocab[:10])


In [ ]:
## 빈도수가 낮은 단어들을 배제
## 등장 빈도수가 3회 미만인 단어들 확인

threshold = 3
total_cnt = len(word_counts) # 단어의 수
rare_cnt = 0 # 등장 빈도수가 threshold보다 작은 단어의 개수를 카운트
total_freq = 0 # 훈련 데이터의 전체 단어 빈도수 총 합
rare_freq = 0 # 등장 빈도수가 threshold보다 작은 단어의 등장 빈도수의 총 합

# 단어와 빈도수의 쌍(pair)을 key와 value로 받는다.
for key, value in word_counts.items():
    total_freq = total_freq + value

    # 단어의 등장 빈도수가 threshold보다 작으면
    if(value < threshold):
        rare_cnt = rare_cnt + 1
        rare_freq = rare_freq + value

print('단어 집합(vocabulary)의 크기 :',total_cnt)
print('등장 빈도가 %s번 이하인 희귀 단어의 수: %s'%(threshold - 1, rare_cnt))
print("단어 집합에서 희귀 단어의 비율:", (rare_cnt / total_cnt)*100)
print("전체 등장 빈도에서 희귀 단어 등장 빈도 비율:", (rare_freq / total_freq)*100)


In [ ]:
# 전체 단어 개수 중 빈도수 3 미만인 단어는 제거.
vocab_size = total_cnt - rare_cnt
vocab = vocab[:vocab_size]
print('단어 집합의 크기 :', len(vocab))


In [ ]:
word_to_index = {}
word_to_index['<PAD>'] = 0 # 패딩 토큰: 문장 길이를 맞추기 위해 사용
word_to_index['<UNK>'] = 1 # 모르는 단어에 대응하기 위함

In [ ]:
for index, word in enumerate(vocab) :
  word_to_index[word] = index + 2


In [ ]:
vocab_size = len(word_to_index)
print('패딩 토큰과 UNK 토큰을 고려한 단어 집합의 크기 :', vocab_size)


In [ ]:
print('단어 <PAD>와 맵핑되는 정수 :', word_to_index['<PAD>'])
print('단어 <UNK>와 맵핑되는 정수 :', word_to_index['<UNK>'])
print('단어 영화와 맵핑되는 정수 :', word_to_index['영화'])


#### 5) 정수 인코딩

In [ ]:
def texts_to_sequences(tokenized_X_data, word_to_index):
  encoded_X_data = []
  for sent in tokenized_X_data:
    index_sequences = []
    for word in sent:
      try:
          index_sequences.append(word_to_index[word])
      except KeyError:
          index_sequences.append(word_to_index['<UNK>'])
    encoded_X_data.append(index_sequences)
  return encoded_X_data


In [ ]:
encoded_X_train = texts_to_sequences(X_train, word_to_index)
encoded_X_test = texts_to_sequences(X_test, word_to_index)

In [ ]:
# 상위 샘플 2개 출력
for sent in encoded_X_train[:2]:
  print(sent)

In [ ]:
index_to_word = {}
for key, value in word_to_index.items():
    index_to_word[value] = key

In [ ]:
decoded_sample = [index_to_word[word] for word in encoded_X_train[0]]
print('기존의 첫번째 샘플 :', X_train[0])
print('복원된 첫번째 샘플 :', decoded_sample)

#### 6) 패딩(Padding)

* 패딩: 서로 다른 길이의 샘플들의 길이를 동일하게 맞춰주는 작업

In [ ]:
print('리뷰의 최대 길이 :',max(len(review) for review in encoded_X_train))
print('리뷰의 평균 길이 :',sum(map(len, encoded_X_train))/len(encoded_X_train))
plt.hist([len(review) for review in encoded_X_train], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()


In [ ]:
def below_threshold_len(max_len, nested_list):
  count = 0
  for sentence in nested_list:
    if(len(sentence) <= max_len):
        count = count + 1
  print('전체 샘플 중 길이가 %s 이하인 샘플의 비율: %s'%(max_len, (count / len(nested_list))*100))

In [ ]:
max_len = 30
below_threshold_len(max_len, X_train)

In [ ]:
def pad_sequences(sentences, max_len):
  features = np.zeros((len(sentences), max_len), dtype=int)
  for index, sentence in enumerate(sentences):
    if len(sentence) != 0:
      features[index, :len(sentence)] = np.array(sentence)[:max_len]
  return features

padded_X_train = pad_sequences(encoded_X_train, max_len=max_len)
padded_X_test = pad_sequences(encoded_X_test, max_len=max_len)

print('훈련 데이터의 크기 :', padded_X_train.shape)
print('테스트 데이터의 크기 :', padded_X_test.shape)

In [ ]:
print('첫번째 샘플의 길이 :', len(padded_X_train[0]))
print('첫번째 샘플 :', padded_X_train[0])

### 2. LSTM을 이용한 네이버 영화 리뷰 분류 모델

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("cpu와 cuda 중 다음 기기로 학습함:", device)

#### 1) 레이블 정리

In [ ]:
y_train = train_data['label'].to_numpy()
y_test = test_data['label'].to_numpy()

In [ ]:
train_label_tensor = torch.tensor(y_train)
test_label_tensor = torch.tensor(y_test)
print(train_label_tensor[:5])


In [ ]:
print(f"학습 데이터 개수: {len(y_train)}, 테스트 데이터 개수: {len(y_test)}")

#### 2) 워드 임베딩(Word Embedding)

**원-핫 인코딩의 한계**
1. 단어의 개수가 늘어날수록, 벡터를 저장하기 위해 필요한 공간이 계속 늘어남
2. 단어 간의 유사도를 표현하지 못함
    - 검색 시스템 등에서 심각한 문제 발생: '삿포로 숙소' 검색 시 '삿포로 게스트 하우스', '삿포로 료칸', '삿포로 호텔' 등에 대한 결과도 나와야 함
##### 
**희소 표현과 밀집 표현**
1. 희소 표현(Sparse Representation)
    * 벡터 또는 행렬의 값의 대부분이 0으로 표현되는 방법
    * 원-핫 벡터는 희소 벡터(Sparse vector)
2. 밀집 표현(Dense Representation)
    * 희소 표현의 반대
    * 사용자가 설정한 값으로 모든 단어의 벡터 표현의 차원을 맞춤

#####
**워드 임베딩**
* 워드 임베딩(Word embedding): 단어를 밀집 벡터(Dense vector)의 형태로 표현하는 방법
    - LSA, Word2Vec, FastText, Glove
* 임베딩 벡터(Embedding vector): 임베딩 과정을 통해 결과로 나온 밀집 벡터

#### 3) 모델 구조

* 단어 벡터의 차원 = 100
* 문장 길이 = 30
* 배치 크기 = 32
* LSTM의 은닉층의 크기 = 128
* 분류하고자 하는 카테고리 개수 = 2개

(배치 크기, 문장 길이) -> 임베딩 층 -> (배치 크기, 문장 길이, 단어 벡터 차원) -> LSTM(Classification model) -> (배치 크기, 은닉층의 크기) -> Softmax -> (배치 크기, 분류 개수)

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (batch_size, seq_length)
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim)

        # LSTM은 (hidden state, cell state)의 튜플을 반환합니다
        lstm_out, (hidden, cell) = self.lstm(embedded)  # lstm_out: (batch_size, seq_length, hidden_dim), hidden: (1, batch_size, hidden_dim)

        last_hidden = hidden.squeeze(0)  # (batch_size, hidden_dim)
        logits = self.fc(last_hidden)  # (batch_size, output_dim)
        return logits


In [ ]:
encoded_train = torch.tensor(padded_X_train).to(torch.int64)
train_dataset = torch.utils.data.TensorDataset(encoded_train, train_label_tensor)
train_dataloader = torch.utils.data.DataLoader(train_dataset, shuffle=True, batch_size=32)

encoded_test = torch.tensor(padded_X_test).to(torch.int64)
test_dataset = torch.utils.data.TensorDataset(encoded_test, test_label_tensor)
test_dataloader = torch.utils.data.DataLoader(test_dataset, shuffle=True, batch_size=1)


In [ ]:
total_batch = len(train_dataloader)
print('총 배치의 수 : {}'.format(total_batch))


In [ ]:
embedding_dim = 100
hidden_dim = 128
output_dim = 2
learning_rate = 0.01
num_epochs = 10

model = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)
model.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 3. 평가 코드 작성

In [ ]:
def calculate_accuracy(logits, labels):
    # _, predicted = torch.max(logits, 1)
    predicted = torch.argmax(logits, dim=1)
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    accuracy = correct / total
    return accuracy

### 4. 학습

In [ ]:
num_epochs = 5

# 학습
for epoch in range(num_epochs):
    train_loss = 0
    train_correct = 0
    train_total = 0
    model.train()
    for batch_X, batch_y in train_dataloader:
        # Forward pass
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        # batch_X.shape == (batch_size, max_len)
        logits = model(batch_X)

        # Compute loss
        loss = criterion(logits, batch_y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate training accuracy and loss
        train_loss += loss.item()
        train_correct += calculate_accuracy(logits, batch_y) * batch_y.size(0)
        train_total += batch_y.size(0)

    train_accuracy = train_correct / train_total
    train_loss /= len(train_dataloader)

    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}')


### 5. 모델 테스트

In [ ]:
index_to_tag = {0 : '부정', 1 : '긍정'}

def predict(text, model, word_to_index, index_to_tag):
    # Set the model to evaluation mode
    model.eval()

    # Tokenize the input text
    tokens = okt.morphs(text) # 토큰화
    tokens = [word for word in tokens if not word in stopwords] # 불용어 제거
    token_indices = [word_to_index.get(token, 1) for token in tokens]

    # Convert tokens to tensor
    input_tensor = torch.tensor([token_indices], dtype=torch.long).to(device)  # (1, seq_length)

    # Pass the input tensor through the model
    with torch.no_grad():
        logits = model(input_tensor)  # (1, output_dim)

    # Get the predicted class index
    predicted_index = torch.argmax(logits, dim=1)

    # Convert the predicted index to its corresponding tag
    predicted_tag = index_to_tag[predicted_index.item()]

    return predicted_tag


In [ ]:
test_input = "이 영화 개꿀잼 ㅋㅋㅋ"
predict(test_input, model, word_to_index, index_to_tag)

In [ ]:
test_input = "이딴게 영화냐 ㅉㅉ"
predict(test_input, model, word_to_index, index_to_tag)


In [ ]:
test_input = "감독 뭐하는 놈이냐?"
predict(test_input, model, word_to_index, index_to_tag)


In [ ]:
test_input = "와 개쩐다 정말 세계관 최강자들의 영화다"
predict(test_input, model, word_to_index, index_to_tag)
